# Phase 3: Evaluation Suite
This notebook benchmarks the **Multi-Start Global Optimization** approach against the **cVAE Generative Modeling** approach.

> **⚠ IMPORTANT:** Before running this suite, you MUST fully run the `cVAE_Generative_Inverse.ipynb` notebook to train and save the `cvae_generator.pth` model!


In [1]:
import os
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.cluster import KMeans

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cpu


In [2]:
# ---------------------------
# 1. LOAD DATA & FROZEN MODELS
# ---------------------------
file_path = '../Data/FINAL_4CLASSES.csv'
df = pd.read_csv(file_path, encoding='utf-8', engine='python')

column_mapping = {
    'Gain(db)': 'Gain(dB)', 'Gain': 'Gain(dB)', 'gain': 'Gain(dB)',
    'Bandwidth': 'Bandwidth(Hz)', 'bandwidth': 'Bandwidth(Hz)',
    'GBW': 'GBW(MHz)', 'gbw': 'GBW(MHz)',
    'Power': 'Power(uW)', 'power': 'Power(uW)',
    'PM': 'PM(degree)', 'PhaseMargin': 'PM(degree)',
    'GM': 'GM(dB)', 'PSRR': 'PSRR(dB)', 'SlewRate': 'SlewRate (V/us)', 
    'SlewRate(V/µs)': 'SlewRate (V/us)', 'CMRR': 'CMRR(dB)'
}
df.rename(columns={k: v for k, v in column_mapping.items() if k in df.columns}, inplace=True)

df['Idc(uA)'] = 130.0
df['Length(um)'] = 0.18
df['CL(pF)'] = 10.0
df['CC(pF)'] = 55.0

FEATURE_COLUMNS = ['Temperature(°)', 'W12(um)', 'W34(um)', 'W58(um)', 'W6(um)', 'W7(um)', 'Idc(uA)', 'Length(um)', 'CC(pF)', 'CL(pF)']
REGRESSION_TARGETS = ['Gain(dB)', 'Bandwidth(Hz)', 'GBW(MHz)', 'Power(uW)', 'PM(degree)', 'GM(dB)', 'PSRR(dB)', 'SlewRate (V/us)', 'CMRR(dB)']

scaler_X = joblib.load('scaler_X.pkl')
scaler_y_reg = joblib.load('scaler_y_reg.pkl')
le = joblib.load('label_encoder.pkl')

width_names = ['W12(um)', 'W34(um)', 'W58(um)', 'W6(um)', 'W7(um)']
width_indices = [FEATURE_COLUMNS.index(col) for col in width_names]
fixed_indices = [FEATURE_COLUMNS.index(col) for col in FEATURE_COLUMNS if col not in width_names]

mu_widths_1d = torch.tensor(scaler_X.mean_[width_indices], dtype=torch.float32, device=device)
std_widths_1d = torch.tensor(scaler_X.scale_[width_indices], dtype=torch.float32, device=device)

# --- LOAD PINN ---
class PINN(nn.Module):
    def __init__(self, input_dim, hidden_dims, n_reg_outputs, n_classes, dropout_rate):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        self.backbone = nn.Sequential(*layers)
        self.regression_head = nn.Linear(prev_dim, n_reg_outputs)
        self.classification_head = nn.Linear(prev_dim, n_classes)

    def forward(self, x):
        shared = self.backbone(x)
        return self.regression_head(shared), self.classification_head(shared)

pinn = PINN(len(FEATURE_COLUMNS), [128, 128, 128, 128], len(REGRESSION_TARGETS), len(le.classes_), 0.047).to(device)
pinn.load_state_dict(torch.load('pinn_model.pth', map_location=device))
pinn.eval()
print("- PINN Forward Model Loaded.")

# --- LOAD cVAE ---
class cVAE(nn.Module):
    def __init__(self, width_dim=5, fixed_dim=5, target_dim=9, latent_dim=3, hidden_dims=[64, 64]):
        super().__init__()
        self.latent_dim = latent_dim
        cond_dim = fixed_dim + target_dim
        
        enc_dim = width_dim + cond_dim
        enc_layers = []
        for h in hidden_dims:
            enc_layers.extend([nn.Linear(enc_dim, h), nn.ReLU()])
            enc_dim = h
        self.encoder = nn.Sequential(*enc_layers)
        self.fc_mu = nn.Linear(enc_dim, latent_dim)
        self.fc_var = nn.Linear(enc_dim, latent_dim)
        
        dec_dim = latent_dim + cond_dim
        dec_layers = []
        for h in hidden_dims:
            dec_layers.extend([nn.Linear(dec_dim, h), nn.ReLU()])
            dec_dim = h
        dec_layers.append(nn.Linear(dec_dim, width_dim))
        self.decoder = nn.Sequential(*dec_layers)
        
    def encode(self, x, cond):
        h = self.encoder(torch.cat([x, cond], dim=-1))
        return self.fc_mu(h), self.fc_var(h)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    def decode(self, z, cond):
        return self.decoder(torch.cat([z, cond], dim=-1))
    def forward(self, x, cond):
        mu, logvar = self.encode(x, cond)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z, cond)
        return x_recon, mu, logvar

cvae = cVAE(latent_dim=3).to(device)
if os.path.exists('cvae_generator.pth'):
    cvae.load_state_dict(torch.load('cvae_generator.pth', map_location=device))
    print("- Generative cVAE Loaded.")
else:
    print("WARNING: cvae_generator.pth not found! Generative evaluation will fail.")
cvae.eval()


- PINN Forward Model Loaded.
- Generative cVAE Loaded.


cVAE(
  (encoder): Sequential(
    (0): Linear(in_features=19, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
  )
  (fc_mu): Linear(in_features=64, out_features=3, bias=True)
  (fc_var): Linear(in_features=64, out_features=3, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=17, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=5, bias=True)
  )
)

## Testing Framework Engines
We define generic functions to run both algorithms so we can measure latency, diversity, and accuracy cleanly.


In [3]:
def run_multi_start(target_metrics, operating_conditions, num_solutions=5):
    start_t = time.time()
    target_array = np.array([[target_metrics[col] for col in REGRESSION_TARGETS]])
    target_scaled = scaler_y_reg.transform(target_array)
    target_tensor = torch.tensor(target_scaled, dtype=torch.float32, device=device)

    N_STARTS = 1000
    noise_scale = 1.5
    inv_widths_raw = (mu_widths_1d + torch.randn((N_STARTS, len(width_names)), device=device) * noise_scale).clone().detach().requires_grad_(True)

    fake_dict = {**operating_conditions, **{k:0 for k in width_names}}
    initial_scaled = scaler_X.transform(np.array([[fake_dict[col] for col in FEATURE_COLUMNS]]))
    fixed_scaled_vals = torch.tensor(initial_scaled[0, fixed_indices], dtype=torch.float32, device=device).unsqueeze(0).repeat(N_STARTS, 1)

    inv_optimizer = optim.Adam([inv_widths_raw], lr=0.1)  
    inv_epochs = 1500

    for epoch in range(1, inv_epochs + 1):
        inv_optimizer.zero_grad()
        widths_phys = F.softplus(inv_widths_raw)
        widths_scaled = (widths_phys - mu_widths_1d) / std_widths_1d
        full_scaled_input = torch.zeros((N_STARTS, len(FEATURE_COLUMNS)), dtype=torch.float32, device=device)
        full_scaled_input[:, width_indices] = widths_scaled
        full_scaled_input[:, fixed_indices] = fixed_scaled_vals
        
        pred_reg, _ = pinn(full_scaled_input)
        loss_batch = F.mse_loss(pred_reg, target_tensor.expand_as(pred_reg), reduction='none').mean(dim=1)
        loss_batch.mean().backward()
        inv_optimizer.step()
        
    latency = time.time() - start_t
    
    final_losses = loss_batch.detach().cpu().numpy()
    final_widths_phys = F.softplus(inv_widths_raw).detach().cpu().numpy()
    
    threshold = 0.05
    success_mask = final_losses < threshold
    valid_w = final_widths_phys[success_mask]
    
    num_found = len(valid_w)
    
    # Extract distinct solutions
    solutions = []
    if num_found > 0:
        K = min(num_solutions, num_found)
        kmeans = KMeans(n_clusters=K, random_state=42).fit(valid_w)
        for cluster_id in range(K):
            cluster_indices = np.where(kmeans.labels_ == cluster_id)[0]
            best_idx = cluster_indices[np.argmin(final_losses[success_mask][cluster_indices])]
            solutions.append(valid_w[best_idx])
            
    return np.array(solutions), num_found, latency


def run_cvae_generator(target_metrics, operating_conditions, num_solutions=5):
    start_t = time.time()
    
    target_array = np.array([[target_metrics[col] for col in REGRESSION_TARGETS]])
    target_scaled = scaler_y_reg.transform(target_array)[0]
    
    fake_dict = {**operating_conditions, **{k:0 for k in width_names}}
    initial_scaled = scaler_X.transform(np.array([[fake_dict[col] for col in FEATURE_COLUMNS]]))
    fixed_scaled = initial_scaled[0, fixed_indices]
    
    cond_fixed = torch.tensor(fixed_scaled, dtype=torch.float32, device=device).unsqueeze(0).repeat(num_solutions, 1)
    cond_target = torch.tensor(target_scaled, dtype=torch.float32, device=device).unsqueeze(0).repeat(num_solutions, 1)
    cond = torch.cat([cond_fixed, cond_target], dim=-1)
    
    with torch.no_grad():
        z = torch.randn((num_solutions, cvae.latent_dim), device=device)
        recon_w_scaled = cvae.decode(z, cond)
        widths_unscaled = (recon_w_scaled * std_widths_1d + mu_widths_1d).cpu().numpy()
        
    latency = time.time() - start_t
    return widths_unscaled, num_solutions, latency


def evaluate_solutions(solutions_phys, target_metrics, operating_conditions):
    if len(solutions_phys) == 0:
        return 0, 0
        
    # 1. Diversity (Average Euclidean pairwise distance)
    dist = 0
    pairs = 0
    for i in range(len(solutions_phys)):
        for j in range(i+1, len(solutions_phys)):
            dist += np.linalg.norm(solutions_phys[i] - solutions_phys[j])
            pairs += 1
    avg_diversity = dist / pairs if pairs > 0 else 0
    
    # 2. PINN Yield Accuracy (Count how many match targets within 5%)
    target_array = np.array([target_metrics[col] for col in REGRESSION_TARGETS])
    
    fake_dict = {**operating_conditions, **{k:0 for k in width_names}}
    initial_scaled = scaler_X.transform(np.array([[fake_dict[col] for col in FEATURE_COLUMNS]]))
    fixed_scaled = initial_scaled[0, fixed_indices]
    
    w_tensor = torch.tensor(solutions_phys, dtype=torch.float32, device=device)
    w_scaled = (w_tensor - mu_widths_1d) / std_widths_1d
    
    full_in = torch.zeros((len(solutions_phys), len(FEATURE_COLUMNS)), dtype=torch.float32, device=device)
    full_in[:, width_indices] = w_scaled
    full_in[:, fixed_indices] = torch.tensor(fixed_scaled, dtype=torch.float32, device=device).unsqueeze(0).repeat(len(solutions_phys), 1)
    
    pinn.eval()
    with torch.no_grad():
        pred_scaled, _ = pinn(full_in)
        pred_unscaled = scaler_y_reg.inverse_transform(pred_eval.cpu().numpy())
    
    # Mean Absolute Percentage Error across all metrics and solutions
    mape_scores = np.mean(np.abs(pred_unscaled - target_array) / (np.abs(target_array) + 1e-8), axis=1) * 100
    successes = sum(1 for m in mape_scores if m < 5.0)  # less than 5% avg error
    yield_rate = (successes / len(solutions_phys)) * 100
    
    return avg_diversity, yield_rate, np.mean(mape_scores)


## Bulk Test Execution
We randomly select 10 target samples and run both engines.


In [4]:
np.random.seed(42)
test_size = 10
test_samples = df.sample(n=test_size, random_state=42)

results_ms = []
results_vae = []

print(f"Benchmarking {test_size} random targets from dataset...\n")

for idx, sample_row in test_samples.iterrows():
    t_metrics = {col: sample_row[col] for col in REGRESSION_TARGETS}
    o_conds = {
        'Temperature(°)': sample_row['Temperature(°)'], 'Idc(uA)': sample_row['Idc(uA)'],
        'Length(um)': sample_row['Length(um)'], 'CC(pF)': sample_row['CC(pF)'], 'CL(pF)': sample_row['CL(pF)']
    }
    
    # 1. Multi Start
    sol_ms, valid_count_ms, lat_ms = run_multi_start(t_metrics, o_conds, num_solutions=5)
    
    # Needs a hack inside eval function fix scope error, let's redefine evaluate internally if used globally
    # In python notebook this works out of the box. 
    
    # Wait, lets fix the eval function for predict.
    def compute_metrics(solutions_phys):
        if len(solutions_phys) == 0: return 0, 0, 100
        dist, pairs = 0, 0
        for i in range(len(solutions_phys)):
            for j in range(i+1, len(solutions_phys)):
                dist += np.linalg.norm(solutions_phys[i] - solutions_phys[j])
                pairs += 1
        div = dist/pairs if pairs > 0 else 0
        
        target_array = np.array([t_metrics[col] for col in REGRESSION_TARGETS])
        fake_dict = {**o_conds, **{k:0 for k in width_names}}
        init_scaled = scaler_X.transform(np.array([[fake_dict[col] for col in FEATURE_COLUMNS]]))
        fix_sc = init_scaled[0, fixed_indices]
        
        w_t = torch.tensor(solutions_phys, dtype=torch.float32, device=device)
        w_s = (w_t - mu_widths_1d) / std_widths_1d
        
        fi = torch.zeros((len(solutions_phys), len(FEATURE_COLUMNS)), dtype=torch.float32, device=device)
        fi[:, width_indices] = w_s
        fi[:, fixed_indices] = torch.tensor(fix_sc, dtype=torch.float32, device=device).unsqueeze(0).repeat(len(solutions_phys), 1)
        
        pinn.eval()
        with torch.no_grad():
            pr_scaled, _ = pinn(fi)
            pr_unscaled = scaler_y_reg.inverse_transform(pr_scaled.cpu().numpy())
            
        m_scores = np.mean(np.abs(pr_unscaled - target_array) / (np.abs(target_array) + 1e-8), axis=1) * 100
        suc = sum(1 for m in m_scores if m < 5.0)
        return div, (suc/len(solutions_phys)) * 100, np.mean(m_scores)
    
    div_ms, y_ms, mape_ms = compute_metrics(sol_ms)
    results_ms.append({'lat': lat_ms, 'div': div_ms, 'yield': y_ms, 'mape': mape_ms})
    
    # 2. VAE Gen
    if os.path.exists('cvae_generator.pth'):
        sol_vae, _, lat_vae = run_cvae_generator(t_metrics, o_conds, num_solutions=5)
        div_vae, y_vae, mape_vae = compute_metrics(sol_vae)
        results_vae.append({'lat': lat_vae, 'div': div_vae, 'yield': y_vae, 'mape': mape_vae})

print("\n==================================")
print(" BENCHMARK RESULTS AVERAGES")
print("==================================")
if len(results_ms) > 0:
    print(f"Multi-Start Optimizer (Gradient Descent Based):")
    print(f" - Latency per target: {np.mean([r['lat'] for r in results_ms]):.3f} sec")
    print(f" - Geometrical Diversity: {np.mean([r['div'] for r in results_ms]):.2f} um unit-distance")
    print(f" - Performance Yield (<5% error block): {np.mean([r['yield'] for r in results_ms]):.1f}%")
    print(f" - Average Output Error (MAPE): {np.mean([r['mape'] for r in results_ms]):.2f}%")

if len(results_vae) > 0:
    print(f"\nGenerative cVAE (Instant Feedforward Inference):")
    print(f" - Latency per target: {np.mean([r['lat'] for r in results_vae]):.4f} sec")
    print(f" - Geometrical Diversity: {np.mean([r['div'] for r in results_vae]):.2f} um unit-distance")
    print(f" - Performance Yield (<5% error block): {np.mean([r['yield'] for r in results_vae]):.1f}%")
    print(f" - Average Output Error (MAPE): {np.mean([r['mape'] for r in results_vae]):.2f}%")
else:
    print("\n[Generative Test Skipped: cvae_generator.pth missing]")


Benchmarking 10 random targets from dataset...


 BENCHMARK RESULTS AVERAGES
Multi-Start Optimizer (Gradient Descent Based):
 - Latency per target: 7.176 sec
 - Geometrical Diversity: 13.56 um unit-distance
 - Performance Yield (<5% error block): 100.0%
 - Average Output Error (MAPE): 1.93%

Generative cVAE (Instant Feedforward Inference):
 - Latency per target: 0.0006 sec
 - Geometrical Diversity: 0.15 um unit-distance
 - Performance Yield (<5% error block): 100.0%
 - Average Output Error (MAPE): 1.80%
